In [91]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from notebookutils import *
from pyspark.sql import functions as F
from delta.tables import *
from notebookutils import mssparkutils

StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, 93, Finished, Available, Finished, True)

In [92]:
tables_str = tables_str
bronze = bronze
lakehouse = lakehouse
workspace = workspace
silver = silver

# tables_str = "Patient,Encounter,Observation,Condition"
# bronze = "bronze"
# silver = "silver"
# workspace = "Fabric_Assignment_Synapx"
# lakehouse = "FHIR_Lakehouse"

Tables = [t.strip() for t in tables_str.split(",")]

StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, 94, Finished, Available, Finished, True)

NameError: name 'raw' is not defined

In [ ]:
table = None
for i in Tables:
    if i.lower() == "condition":
        table = i
        break

if table is None:
    mssparkutils.notebook.exit("Exiting notebook: Table is not condition")

print(table + " table is present ")

StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, -1, Cancelled, , Cancelled, True)

In [ ]:
bronze_path = f"Files/{bronze}/{table.lower()}"
silver_path = f"Files/{silver}/{table.lower()}"

# Check if silver path exists
silver_exists = mssparkutils.fs.exists(silver_path)

# Read bronze data
df = spark.read.format("delta").load(bronze_path)

# Apply filter only if silver exists
if silver_exists:
    df = df.filter(
        col("load_date") >= date_sub(current_date(), 4)
    )


StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, -1, Cancelled, , Cancelled, True)

In [ ]:
# -----------------------------
# correct field (entry inside struct)
# -----------------------------
df1 = df.withColumn("entry_block", col("raw_json.entry"))

# -----------------------------
# Split entries (since it's a big string list)
# -----------------------------
df2 = df1.withColumn(
    "entry_array",
    split(col("entry_block"), r"\},\s*\{")
)

# -----------------------------
#  Explode entries
# -----------------------------
df3 = df2.withColumn("entry", explode(col("entry_array")))

# -----------------------------
# Extract key fields
# -----------------------------

df3 = df3.withColumn(
    "resource_id",
    regexp_extract(col("entry"), r"id=([^,}]+)", 1)
)

df3 = df3.withColumn(
    "resource_type",
    regexp_extract(col("entry"), r"resourceType=([^,}]+)", 1)
)

df3 = df3.withColumn(
    "condition_text",
    regexp_extract(col("entry"), r"text=([^,}]+)", 1)
)

df3 = df3.withColumn(
    "patient_ref",
    regexp_extract(col("entry"), r"subject=\{reference=([^,}]+)", 1)
)

df3 = df3.withColumn(
    "encounter_ref",
    regexp_extract(col("entry"), r"encounter=\{reference=([^,}]+)", 1)
)

df3 = df3.withColumn(
    "asserter",
    regexp_extract(col("entry"), r"asserter=\{reference=([^,}]+)", 1)
)

df3 = df3.withColumn(
    "last_updated",
    regexp_extract(col("entry"), r"lastUpdated=([^,}]+)", 1)
)

silver_condition = df3.select(
    "resource_id",
    "resource_type",
    "condition_text",
    "patient_ref",
    "encounter_ref",
    "asserter",
    "last_updated",
    "load_date",
    "extraction_timestamp"
)


StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, -1, Cancelled, , Cancelled, True)

In [ ]:
# -----------------------------
#  Create IDs FIRST (before drop)
# -----------------------------
silver_condition_fixed = (
    silver_condition
    .withColumn(
        "patient_id",
        trim(split(col("patient_ref"), "/")[1])
    )
    .withColumn(
        "encounter_id",
        trim(split(col("encounter_ref"), "/")[1])
    )
    .withColumn(
        "Practitioner_id",
        trim(split(col("asserter"), "/")[1])
    )
)

# -----------------------------
# Drop old columns safely
# -----------------------------
silver_condition_fixed = silver_condition_fixed.drop(
    "patient_ref",
    "encounter_ref",
    "asserter"
)

# -----------------------------
#  Fix timestamp type (important for ordering)
# -----------------------------
silver_condition_fixed = silver_condition_fixed.withColumn(
    "last_updated",
    to_timestamp(col("last_updated"))
)

# -----------------------------
#  Display sorted output
# -----------------------------
display(
    silver_condition_fixed.orderBy(col("last_updated")).limit(100)
)

silver_condition = silver_condition_fixed

StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, -1, Cancelled, , Cancelled, True)

In [ ]:
# -----------------------------
# Step 1: Ensure timestamp type
# -----------------------------
silver_condition = silver_condition.withColumn(
    "last_updated",
    to_timestamp(col("last_updated"))
)

# -----------------------------
# Step 2: Define window
# -----------------------------
window = Window.partitionBy("resource_id").orderBy(
    col("last_updated").desc()
)

# -----------------------------
# Step 3: Deduplicate using row_number
# -----------------------------
silver_condition = (
    silver_condition
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
)

# -----------------------------
# Step 4: Display result (Fabric-safe)
# -----------------------------
display(silver_condition.limit(5))

StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, -1, Cancelled, , Cancelled, True)

In [ ]:
staging_df = silver_condition

staging_df = staging_df \
    .withColumn("effective_start_date", col("last_updated").cast("timestamp")) \
    .withColumn("effective_end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, -1, Cancelled, , Cancelled, True)

In [ ]:
silver_path = f"Files/{silver}/{table.lower()}"

if not mssparkutils.fs.exists(silver_path):
    staging_df.write.format("delta").mode("overwrite").save(silver_path)

    mssparkutils.notebook.exit("Exiting notebook: First time load completed")
else:
    print("Table exists")

StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, -1, Cancelled, , Cancelled, True)

In [ ]:

staging_df.createOrReplaceTempView("staging_condition")

StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, -1, Cancelled, , Cancelled, True)

In [ ]:
query = f"""
MERGE INTO delta.`{silver_path}` AS tgt
USING staging_condition AS src
ON tgt.resource_id = src.resource_id
AND tgt.resource_type = src.resource_type
AND tgt.is_current = true

WHEN MATCHED AND (
    tgt.condition_text <> src.condition_text OR
    tgt.patient_id <> src.patient_id OR
    tgt.encounter_id <> src.encounter_id OR
    tgt.Practitioner_id <> src.Practitioner_id
)
THEN UPDATE SET
    tgt.is_current = false,
    tgt.effective_end_date = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
    resource_id,
    resource_type,
    condition_text,
    Practitioner_id,
    last_updated,
    load_date,
    extraction_timestamp,
    patient_id,
    encounter_id,
    effective_start_date,
    effective_end_date,
    is_current
)
VALUES (
    src.resource_id,
    src.resource_type,
    src.condition_text,
    src.Practitioner_id,
    src.last_updated,
    src.load_date,
    src.extraction_timestamp,
    src.patient_id,
    src.encounter_id,
    current_timestamp(),
    NULL,
    true
)
"""
query1 = f"""
INSERT INTO delta.`{silver_path}`
SELECT 
    src.resource_id,
    src.resource_type,
    src.condition_text,
    src.Practitioner_id,
    src.last_updated,
    src.load_date,
    src.extraction_timestamp,
    src.patient_id,
    src.encounter_id,
    current_timestamp() AS effective_start_date,
    NULL AS effective_end_date,
    true AS is_current
FROM staging_condition src
JOIN delta.`{silver_path}` tgt
    ON tgt.resource_id = src.resource_id
    AND tgt.resource_type = src.resource_type
WHERE 
    tgt.is_current = false
AND (
    tgt.condition_text <> src.condition_text OR
    tgt.patient_id <> src.patient_id OR
    tgt.encounter_id <> src.encounter_id OR
    tgt.Practitioner_id <> src.Practitioner_id
)
AND NOT EXISTS (
    SELECT 1
    FROM delta.`{silver_path}` t2
    WHERE t2.resource_id = src.resource_id
      AND t2.resource_type = src.resource_type
      AND t2.is_current = true
)
"""
spark.sql(query)
spark.sql(query1)

StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, -1, Cancelled, , Cancelled, True)

In [ ]:
df = spark.sql(f"DESCRIBE HISTORY delta.`{silver_path}`") \
    .orderBy("version", ascending=False) \
    .limit(2)

display(df)

StatementMeta(, 8faf40b6-811f-43f8-a297-394c47479c00, -1, Cancelled, , Cancelled, True)